# Single-cell and Spatial Transcriptomics Data Integration (Simplified Version)

This script uses Harmony algorithm to integrate single-cell and spatial transcriptomics data from lamprey, and performs batch correction and clustering analysis.

In [ ]:
library(Seurat)
library(dplyr)
library(harmony)
library(ggplot2)

set.seed(111)

## 1. Data Loading

Load the integrated single-cell and spatial transcriptomics data (RDS format)

In [ ]:
data <- readRDS("/data/users/niuzhiwei1/online/08七鳃鳗全部细胞整合(包括小脑)_与其他物种整合/02七鳃鳗空间和单细胞NR数据整合/02空间全部细胞与单细胞整合/01七鳃鳗空间全部细胞和单细胞NR数据(包括小脑)整合_去双胞与neuron_glia_新注释_所有基因_未分析.rds")

## 2. Data Preprocessing

Includes:
- Identify highly variable genes (2000 genes)
- Data normalization
- Principal Component Analysis (PCA)

In [ ]:
datanew <- data %>%
  FindVariableFeatures(nfeatures = 2000) %>%
  ScaleData(verbose = FALSE) %>%
  RunPCA(verbose = FALSE)

## 3. Ensure Species Information is Character Type

Harmony requires group.by variable to be character or factor type

In [ ]:
datanew@meta.data$species <- as.character(datanew@meta.data$species)

## 4. Harmony Batch Correction

Use Harmony algorithm to correct technical differences between different species (batches)
- group.by: specify batch variable (species)
- dims.use: use first 20 principal components

In [ ]:
Combine <- RunHarmony(datanew, group.by = "species", dims.use = 1:20)

## 5. Dimensionality Reduction and Clustering

Based on Harmony-corrected coordinates:
- UMAP dimensionality reduction
- Build neighbor graph
- Clustering analysis (resolution = 0.5)

In [ ]:
Combine <- Combine %>%
  RunUMAP(reduction = "harmony", dims = 1:20) %>%
  FindNeighbors(reduction = "harmony", dims = 1:20) %>%
  FindClusters(resolution = 0.5)

## 6. Data Visualization

### 6.1 View Species Distribution

In [ ]:
DimPlot(Combine, group.by = "species")

### 6.2 View Clustering Results

In [ ]:
DimPlot(Combine, label = TRUE)

### 6.3 Split Clustering Results by Species

In [ ]:
DimPlot(Combine, split.by = "species")

### 6.4 View Cell Type Annotations

In [ ]:
DimPlot(Combine, group.by = "class_anno.main", label = TRUE) + NoLegend()

## 7. Subset Analysis

### 7.1 Extract Single-cell Data (Leth_reis_other)

In [ ]:
sc_data <- subset(Combine, species == "Leth_reis_other")
DimPlot(sc_data, reduction = "harmony", group.by = "class_anno.main", label = TRUE) + NoLegend()

### 7.2 Extract Spatial Transcriptomics Data (Leth_reis_spa)

In [ ]:
sp_data <- subset(Combine, species == "Leth_reis_spa")
DimPlot(sp_data, group.by = "region", label = TRUE) + NoLegend()

## 8. Quality Control

### 8.1 View Gene Count Distribution

In [ ]:
VlnPlot(Combine, features = "nFeature_RNA")

### 8.2 Filter Low-quality Cells

For single-cell data, filter out cells with gene count less than 500

In [ ]:
Combine_filt <- subset(Combine, 
  subset = (species == "Leth_reis_other" & nFeature_RNA > 500) | species != "Leth_reis_other")

## 9. Save Results

Save the integrated data as RDS file

In [ ]:
saveRDS(Combine, "sc.sp.les.harmony.rds")